
# Customer Value Integration

## Business Objective

Bring together customer purchasing behaviour, repeat-purchase propensity, and observed merchandise value into a consistent customer intelligence dataset.

## Business Question

How can the business distinguish customers who have generated value from customers who may generate additional value in the future?

## Integration Scope

- Preserve the existing customer intelligence population.
- Integrate validated historical 60-day merchandise value.
- Keep actual customer outcomes separate from predictive scores.
- Validate customer identifiers, population coverage, and monetary totals.
- Prepare a governed dataset for PostgreSQL and Power BI.

## Important Distinction

Historical merchandise value represents observed purchasing outcomes.

Repeat-purchase propensity represents a model prediction.

Experimental predicted monetary value is not yet independently validated and will not be treated as production-ready Customer Lifetime Value.

## Expected Outcome

A consistent customer-level data foundation that supports business reporting, customer-value analysis, and future predictive capabilities.


In [1]:

from pathlib import Path

import pandas as pd
import numpy as np

# Locate the project directory and processed datasets
project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"

# Load existing customer intelligence and historical value datasets
customer_intelligence = pd.read_parquet(
    processed_dir / "customer_intelligence_historical.parquet"
)

customer_value = pd.read_parquet(
    processed_dir / "customer_value_60d_targets.parquet"
)

# Validate customer populations and identifiers
assert len(customer_intelligence) == 78_100
assert len(customer_value) == 78_100

assert customer_intelligence["customer_unique_id"].is_unique
assert customer_value["customer_unique_id"].is_unique

assert customer_intelligence["customer_unique_id"].notna().all()
assert customer_value["customer_unique_id"].notna().all()

# Confirm that both datasets represent identical customers
intelligence_ids = set(
    customer_intelligence["customer_unique_id"]
)

value_ids = set(
    customer_value["customer_unique_id"]
)

population_audit = pd.DataFrame({
    "Validation": [
        "Customer intelligence records",
        "Historical value records",
        "Customers in both datasets",
        "Customers missing from value data",
        "Customers missing from intelligence data"
    ],
    "Count": [
        len(customer_intelligence),
        len(customer_value),
        len(intelligence_ids & value_ids),
        len(intelligence_ids - value_ids),
        len(value_ids - intelligence_ids)
    ]
})

print("CUSTOMER INTEGRATION POPULATION AUDIT")

display(population_audit)

print("\nCUSTOMER INTELLIGENCE COLUMNS")
print(customer_intelligence.columns.tolist())

print("\nHISTORICAL VALUE COLUMNS")
print(customer_value.columns.tolist())


CUSTOMER INTEGRATION POPULATION AUDIT


,Validation,Count
0,Customer intelligence records,78100
1,Historical value records,78100
2,Customers in both datasets,78100
3,Customers missing from value data,0
4,Customers missing from intelligence data,0



CUSTOMER INTELLIGENCE COLUMNS
['customer_unique_id', 'order_id', 'order_purchase_timestamp', 'first_order_item_value', 'repeat_purchase_propensity', 'propensity_segment', 'initial_value_segment', 'customer_profile', 'repeat_purchase_60d']

HISTORICAL VALUE COLUMNS
['customer_unique_id', 'initial_delivered_value', 'subsequent_60d_value', 'total_60d_merchandise_value']


In [4]:

# Integrate customer intelligence with observed historical merchandise value
# Preserve the original customer population and existing classifications

customer_value_integrated = customer_intelligence.merge(
    customer_value,
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

# Validate the integrated customer population
assert len(customer_value_integrated) == 78_100
assert customer_value_integrated["customer_unique_id"].is_unique
assert customer_value_integrated[
    [
        "initial_delivered_value",
        "subsequent_60d_value",
        "total_60d_merchandise_value"
    ]
].notna().all().all()

# Reconcile integrated monetary values with Notebook 09
assert np.isclose(
    customer_value_integrated["subsequent_60d_value"].sum(),
    211_295.19,
    atol=0.01
)

assert np.isclose(
    customer_value_integrated["total_60d_merchandise_value"].sum(),
    10_606_400.87,
    atol=0.01
)

assert np.allclose(
    customer_value_integrated["initial_delivered_value"]
    + customer_value_integrated["subsequent_60d_value"],
    customer_value_integrated["total_60d_merchandise_value"]
)

print("CUSTOMER VALUE INTEGRATION SUCCESSFUL")
print(f"Customers: {len(customer_value_integrated):,}")
print(f"Columns: {customer_value_integrated.shape[1]}")

print("\nMONETARY RECONCILIATION")
print(
    "Initial delivered merchandise value:",
    f"R${customer_value_integrated['initial_delivered_value'].sum():,.2f}"
)
print(
    "Subsequent merchandise value:",
    f"R${customer_value_integrated['subsequent_60d_value'].sum():,.2f}"
)
print(
    "Total observed 60-day merchandise value:",
    f"R${customer_value_integrated['total_60d_merchandise_value'].sum():,.2f}"
)

print("\nINTEGRATED DATASET PREVIEW")
display(customer_value_integrated.head())


CUSTOMER VALUE INTEGRATION SUCCESSFUL
Customers: 78,100
Columns: 12

MONETARY RECONCILIATION
Initial delivered merchandise value: R$10,395,105.68
Subsequent merchandise value: R$211,295.19
Total observed 60-day merchandise value: R$10,606,400.87

INTEGRATED DATASET PREVIEW


,customer_unique_id,order_id,order_purchase_timestamp,first_order_item_value,repeat_purchase_propensity,propensity_segment,initial_value_segment,customer_profile,repeat_purchase_60d,initial_delivered_value,subsequent_60d_value,total_60d_merchandise_value
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27,129.90,0.045277,High Propensity,Higher Initial Value,Higher Value | High Propensity,0,129.90,0.0,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27,18.90,0.018012,Low Propensity,Lower Initial Value,Lower Value | Low Propensity,0,18.90,0.0,18.90
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03,69.00,0.012786,Low Propensity,Lower Initial Value,Lower Value | Low Propensity,0,69.00,0.0,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41,25.99,0.017735,Low Propensity,Lower Initial Value,Lower Value | Low Propensity,0,25.99,0.0,25.99
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42,180.00,0.014317,Low Propensity,Higher Initial Value,Higher Value | Low Propensity,0,180.00,0.0,180.00


In [5]:

# Validate repeat-purchase labels against observed subsequent value

repeat_value_validation = pd.crosstab(
    customer_value_integrated["repeat_purchase_60d"],
    customer_value_integrated["subsequent_60d_value"] > 0,
    rownames=["Repeat purchase label"],
    colnames=["Positive subsequent value"]
)

print("REPEAT PURCHASE AND HISTORICAL VALUE RECONCILIATION")
display(repeat_value_validation)

mismatch_count = (
    customer_value_integrated["repeat_purchase_60d"].astype(bool)
    != (customer_value_integrated["subsequent_60d_value"] > 0)
).sum()

print(f"\nClassification mismatches: {mismatch_count:,}")

assert mismatch_count == 0, (
    "Repeat-purchase labels do not reconcile with observed value."
)

print("Repeat-purchase classification reconciled successfully.")


REPEAT PURCHASE AND HISTORICAL VALUE RECONCILIATION


Positive subsequent value,False,True
Repeat purchase label,,
0,76511,0
1,0,1589



Classification mismatches: 0
Repeat-purchase classification reconciled successfully.


In [6]:

# Evaluate observed repeat purchases and subsequent merchandise value
# across the existing propensity segments

propensity_value_summary = (
    customer_value_integrated
    .groupby("propensity_segment", dropna=False)
    .agg(
        customers=("customer_unique_id", "count"),
        repeat_purchasers=("repeat_purchase_60d", "sum"),
        observed_repeat_rate=("repeat_purchase_60d", "mean"),
        average_predicted_propensity=("repeat_purchase_propensity", "mean"),
        total_subsequent_value=("subsequent_60d_value", "sum"),
        average_subsequent_value=("subsequent_60d_value", "mean"),
        total_60d_value=("total_60d_merchandise_value", "sum")
    )
    .reset_index()
)

propensity_value_summary["customer_share_pct"] = (
    100 * propensity_value_summary["customers"]
    / len(customer_value_integrated)
)

propensity_value_summary["subsequent_value_share_pct"] = (
    100 * propensity_value_summary["total_subsequent_value"]
    / customer_value_integrated["subsequent_60d_value"].sum()
)

propensity_value_summary["observed_repeat_rate_pct"] = (
    100 * propensity_value_summary["observed_repeat_rate"]
)

propensity_value_summary["average_predicted_propensity_pct"] = (
    100 * propensity_value_summary["average_predicted_propensity"]
)

print("OBSERVED CUSTOMER VALUE BY PROPENSITY SEGMENT")

display(
    propensity_value_summary[
        [
            "propensity_segment",
            "customers",
            "customer_share_pct",
            "repeat_purchasers",
            "observed_repeat_rate_pct",
            "average_predicted_propensity_pct",
            "total_subsequent_value",
            "subsequent_value_share_pct",
            "average_subsequent_value"
        ]
    ].round(2)
)

# Verify that segment totals reconcile with the integrated dataset
assert propensity_value_summary["customers"].sum() == 78_100

assert propensity_value_summary["repeat_purchasers"].sum() == 1_589

assert np.isclose(
    propensity_value_summary["total_subsequent_value"].sum(),
    customer_value_integrated["subsequent_60d_value"].sum()
)

print("\nSegment totals reconciled successfully.")


OBSERVED CUSTOMER VALUE BY PROPENSITY SEGMENT


,propensity_segment,customers,customer_share_pct,repeat_purchasers,observed_repeat_rate_pct,average_predicted_propensity_pct,total_subsequent_value,subsequent_value_share_pct,average_subsequent_value
0,High Propensity,15620,20.0,559,3.58,4.11,67254.96,31.83,4.31
1,Low Propensity,39050,50.0,536,1.37,1.53,85712.60,40.57,2.19
2,Medium Propensity,23430,30.0,494,2.11,2.25,58327.63,27.60,2.49



Segment totals reconciled successfully.


## Observed Customer Value by Customer Profile

In [7]:

# Compare observed customer value across the existing customer profiles

profile_value_summary = (
    customer_value_integrated
    .groupby("customer_profile", dropna=False)
    .agg(
        customers=("customer_unique_id", "count"),
        repeat_purchasers=("repeat_purchase_60d", "sum"),
        observed_repeat_rate=("repeat_purchase_60d", "mean"),
        total_initial_value=("initial_delivered_value", "sum"),
        total_subsequent_value=("subsequent_60d_value", "sum"),
        total_60d_value=("total_60d_merchandise_value", "sum"),
        average_60d_value=("total_60d_merchandise_value", "mean"),
        average_subsequent_value=("subsequent_60d_value", "mean")
    )
    .reset_index()
)

profile_value_summary["customer_share_pct"] = (
    100 * profile_value_summary["customers"]
    / len(customer_value_integrated)
)

profile_value_summary["observed_repeat_rate_pct"] = (
    100 * profile_value_summary["observed_repeat_rate"]
)

profile_value_summary["subsequent_value_share_pct"] = (
    100 * profile_value_summary["total_subsequent_value"]
    / customer_value_integrated["subsequent_60d_value"].sum()
)

print("OBSERVED CUSTOMER VALUE BY CUSTOMER PROFILE")

display(
    profile_value_summary[
        [
            "customer_profile",
            "customers",
            "customer_share_pct",
            "repeat_purchasers",
            "observed_repeat_rate_pct",
            "total_initial_value",
            "total_subsequent_value",
            "subsequent_value_share_pct",
            "average_subsequent_value",
            "average_60d_value"
        ]
    ].round(2)
)

# Reconcile profile totals with the validated customer population
assert profile_value_summary["customers"].sum() == 78_100
assert profile_value_summary["repeat_purchasers"].sum() == 1_589

assert np.isclose(
    profile_value_summary["total_60d_value"].sum(),
    customer_value_integrated["total_60d_merchandise_value"].sum()
)

print("\nCustomer profile totals reconciled successfully.")


OBSERVED CUSTOMER VALUE BY CUSTOMER PROFILE


,customer_profile,customers,customer_share_pct,repeat_purchasers,observed_repeat_rate_pct,total_initial_value,total_subsequent_value,subsequent_value_share_pct,average_subsequent_value,average_60d_value
0,Higher Value | High Propensity,7287,9.33,259,3.55,1239690.93,40631.07,19.23,5.58,175.70
1,Higher Value | Low Propensity,20931,26.80,249,1.19,5474097.04,57619.49,27.27,2.75,264.28
2,Higher Value | Medium Propensity,10543,13.50,209,1.98,1932099.74,31855.87,15.08,3.02,186.28
3,Lower Value | High Propensity,7814,10.01,284,3.63,388633.46,22825.82,10.80,2.92,52.66
4,Lower Value | Low Propensity,18117,23.20,285,1.57,786515.20,27958.22,13.23,1.54,44.96
5,Lower Value | Medium Propensity,12766,16.35,282,2.21,574069.31,26342.29,12.47,2.06,47.03
6,Unknown Initial Value,642,0.82,21,3.27,0.00,4062.43,1.92,6.33,6.33



Customer profile totals reconciled successfully.


## Data Quality Audit: Unknown Initial Value

Investigate customers with unknown initial-value classification
before finalising the integrated customer reporting dataset.

In [8]:

# Investigate customers whose initial-value classification is unknown

unknown_initial_value = customer_value_integrated.loc[
    customer_value_integrated["initial_value_segment"].isna()
    | customer_value_integrated["initial_value_segment"].astype(str).str.contains(
        "Unknown", case=False, na=False
    )
].copy()

print("UNKNOWN INITIAL VALUE AUDIT")

audit_summary = pd.Series({
    "Customers": len(unknown_initial_value),
    "Missing first-order item values":
        unknown_initial_value["first_order_item_value"].isna().sum(),
    "Zero first-order item values":
        unknown_initial_value["first_order_item_value"].eq(0).sum(),
    "Missing initial delivered values":
        unknown_initial_value["initial_delivered_value"].isna().sum(),
    "Zero initial delivered values":
        unknown_initial_value["initial_delivered_value"].eq(0).sum(),
    "Customers with subsequent value":
        unknown_initial_value["subsequent_60d_value"].gt(0).sum(),
    "Total subsequent value":
        unknown_initial_value["subsequent_60d_value"].sum()
})

display(audit_summary.to_frame("Result"))

print("\nFIRST-ORDER ITEM VALUE DISTRIBUTION")
display(
    unknown_initial_value["first_order_item_value"].describe()
)

print("\nSAMPLE CUSTOMER RECORDS")
display(
    unknown_initial_value[
        [
            "customer_unique_id",
            "first_order_item_value",
            "initial_value_segment",
            "initial_delivered_value",
            "subsequent_60d_value",
            "repeat_purchase_60d"
        ]
    ].head(10)
)


UNKNOWN INITIAL VALUE AUDIT


,Result
Customers,642.00
Missing first-order item values,642.00
Zero first-order item values,0.00
Missing initial delivered values,0.00
Zero initial delivered values,642.00
Customers with subsequent value,21.00
Total subsequent value,4062.43



FIRST-ORDER ITEM VALUE DISTRIBUTION


count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: first_order_item_value, dtype: float64


SAMPLE CUSTOMER RECORDS


,customer_unique_id,first_order_item_value,initial_value_segment,initial_delivered_value,subsequent_60d_value,repeat_purchase_60d
95,004ad212c650c96c8a48f30855a7bcb8,NaN,Unknown Value,0.0,0.0,0
136,0071f46a072a9ae25bbe4438b15efe9c,NaN,Unknown Value,0.0,0.0,0
189,009b0127b727ab0ba422f6d9604487c7,NaN,Unknown Value,0.0,0.0,0
299,00f0b70fdcb8a6e1671b52a2472bd41f,NaN,Unknown Value,0.0,0.0,0
388,013b083525868a0437652ebf071cda03,NaN,Unknown Value,0.0,0.0,0
625,020d46eecb7bda2b3ab2316a91bb49f6,NaN,Unknown Value,0.0,0.0,0
751,027b5658f8a94a67c3d037047ca849c5,NaN,Unknown Value,0.0,0.0,0
866,02d73d09e28bcf111eb693e80e33cb51,NaN,Unknown Value,0.0,0.0,0
897,02ed27f9cd8f0a7ffc59c7e10b46b9c0,NaN,Unknown Value,0.0,0.0,0
1017,03459d9b3e14feb0e89b0e5bce4f188d,NaN,Unknown Value,0.0,0.0,0


## First-Order Data Availability Investigation

Investigate whether missing initial basket values are associated
with order status, and establish how these customers should be
represented in downstream reporting.

In [9]:

# Investigate original order statuses for customers with unknown initial value

orders = pd.read_csv(
    project_root / "data" / "olist_orders_dataset.csv",
    usecols=["order_id", "order_status"]
)

unknown_order_status = unknown_initial_value.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("FIRST-ORDER STATUS DISTRIBUTION")
display(
    unknown_order_status["order_status"]
    .fillna("Missing order record")
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="customers")
)

print("\nSTATUS AND SUBSEQUENT PURCHASES")
display(
    pd.crosstab(
        unknown_order_status["order_status"].fillna("Missing order record"),
        unknown_order_status["subsequent_60d_value"] > 0,
        rownames=["First-order status"],
        colnames=["Positive subsequent merchandise value"]
    )
)

assert len(unknown_order_status) == 642
assert unknown_order_status["customer_unique_id"].is_unique

print("\nAll 642 customers retained.")


FIRST-ORDER STATUS DISTRIBUTION


,order_status,customers
0,unavailable,554
1,canceled,80
2,created,5
3,invoiced,2
4,shipped,1



STATUS AND SUBSEQUENT PURCHASES


Positive subsequent merchandise value,False,True
First-order status,,
canceled,68,12
created,5,0
invoiced,2,0
shipped,1,0
unavailable,545,9



All 642 customers retained.


In [10]:

# Add original first-order status to the integrated customer dataset

assert orders["order_id"].is_unique

customer_value_integrated = customer_value_integrated.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one"
)

# Create a reporting-friendly indicator based on recorded order status
customer_value_integrated["first_order_delivery_status"] = np.where(
    customer_value_integrated["order_status"].eq("delivered"),
    "Delivered",
    "Not Delivered"
)

# Preserve missing order records as a distinct data-quality condition
customer_value_integrated.loc[
    customer_value_integrated["order_status"].isna(),
    "first_order_delivery_status"
] = "Unknown Status"

print("FIRST-ORDER DELIVERY STATUS")
display(
    customer_value_integrated["first_order_delivery_status"]
    .value_counts(dropna=False)
)

print("\nUNKNOWN INITIAL VALUE RECONCILIATION")

unknown_value_check = customer_value_integrated.loc[
    customer_value_integrated["initial_value_segment"].eq("Unknown Value")
]

display(
    unknown_value_check["first_order_delivery_status"]
    .value_counts(dropna=False)
)

assert len(customer_value_integrated) == 78_100
assert customer_value_integrated["customer_unique_id"].is_unique

assert len(unknown_value_check) == 642

assert (
    unknown_value_check["first_order_delivery_status"] == "Not Delivered"
).all()

print("\nCustomer population and unknown-value classification reconciled.")


FIRST-ORDER DELIVERY STATUS


first_order_delivery_status
Delivered        75599
Not Delivered     2501
Name: count, dtype: int64


UNKNOWN INITIAL VALUE RECONCILIATION


first_order_delivery_status
Not Delivered    642
Name: count, dtype: int64


Customer population and unknown-value classification reconciled.


## Initial-Value Availability by First-Order Status

Examine how first-order delivery status relates to recorded basket
features and observed merchandise value. Preserve the distinction
between missing features, unsuccessful orders, and realised value.

In [11]:

# Audit initial basket-value availability and realised merchandise value
# across first-order delivery classifications

delivery_value_audit = (
    customer_value_integrated
    .groupby("first_order_delivery_status", dropna=False)
    .agg(
        customers=("customer_unique_id", "count"),
        missing_first_order_item_value=(
            "first_order_item_value",
            lambda values: values.isna().sum()
        ),
        available_first_order_item_value=(
            "first_order_item_value",
            lambda values: values.notna().sum()
        ),
        zero_initial_delivered_value=(
            "initial_delivered_value",
            lambda values: values.eq(0).sum()
        ),
        positive_initial_delivered_value=(
            "initial_delivered_value",
            lambda values: values.gt(0).sum()
        ),
        repeat_purchasers=("repeat_purchase_60d", "sum"),
        total_initial_delivered_value=("initial_delivered_value", "sum"),
        total_subsequent_value=("subsequent_60d_value", "sum")
    )
    .reset_index()
)

delivery_value_audit["missing_feature_pct"] = (
    100
    * delivery_value_audit["missing_first_order_item_value"]
    / delivery_value_audit["customers"]
)

print("FIRST-ORDER DELIVERY AND VALUE AVAILABILITY AUDIT")
display(delivery_value_audit.round(2))

print("\nFIRST-ORDER STATUS AND MISSING BASKET VALUE")
display(
    pd.crosstab(
        customer_value_integrated["order_status"].fillna("Missing Status"),
        customer_value_integrated["first_order_item_value"].isna(),
        rownames=["Original first-order status"],
        colnames=["First-order basket value missing"]
    )
)

# Verify that the audit preserves the customer population and values
assert delivery_value_audit["customers"].sum() == 78_100

assert delivery_value_audit["repeat_purchasers"].sum() == 1_589

assert np.isclose(
    delivery_value_audit["total_subsequent_value"].sum(),
    customer_value_integrated["subsequent_60d_value"].sum()
)

print("\nDelivery and customer-value totals reconciled successfully.")


FIRST-ORDER DELIVERY AND VALUE AVAILABILITY AUDIT


,first_order_delivery_status,customers,missing_first_order_item_value,available_first_order_item_value,zero_initial_delivered_value,positive_initial_delivered_value,repeat_purchasers,total_initial_delivered_value,total_subsequent_value,missing_feature_pct
0,Delivered,75599,0,75599,0,75599,1520,10395105.68,197818.14,0.00
1,Not Delivered,2501,642,1859,2501,0,69,0.00,13477.05,25.67



FIRST-ORDER STATUS AND MISSING BASKET VALUE


First-order basket value missing,False,True
Original first-order status,,
approved,2,0
canceled,364,80
created,0,5
delivered,75599,0
invoiced,267,2
processing,293,0
shipped,927,1
unavailable,6,554



Delivery and customer-value totals reconciled successfully.


## Final Customer Intelligence Reporting Dataset

Prepare a validated customer-level dataset combining purchasing
behaviour, predicted repeat-purchase propensity, observed merchandise
value, and first-order delivery status.

In [12]:

# Define the customer-level reporting schema
# Keep predictions separate from observed monetary outcomes

reporting_columns = [
    "customer_unique_id",
    "order_id",
    "order_purchase_timestamp",
    "order_status",
    "first_order_delivery_status",
    "first_order_item_value",
    "repeat_purchase_propensity",
    "propensity_segment",
    "initial_value_segment",
    "customer_profile",
    "repeat_purchase_60d",
    "initial_delivered_value",
    "subsequent_60d_value",
    "total_60d_merchandise_value"
]

customer_reporting = customer_value_integrated[
    reporting_columns
].copy()

# Verify the reporting population and customer-level uniqueness
assert len(customer_reporting) == 78_100
assert customer_reporting["customer_unique_id"].is_unique
assert customer_reporting["customer_unique_id"].notna().all()

# Validate prediction and outcome boundaries
assert customer_reporting[
    "repeat_purchase_propensity"
].between(0, 1).all()

assert customer_reporting[
    [
        "initial_delivered_value",
        "subsequent_60d_value",
        "total_60d_merchandise_value"
    ]
].notna().all().all()

assert (
    customer_reporting[
        [
            "initial_delivered_value",
            "subsequent_60d_value",
            "total_60d_merchandise_value"
        ]
    ] >= 0
).all().all()

# Reconcile financial totals against the validated historical analysis
assert np.isclose(
    customer_reporting["initial_delivered_value"].sum(),
    10_395_105.68,
    atol=0.01
)

assert np.isclose(
    customer_reporting["subsequent_60d_value"].sum(),
    211_295.19,
    atol=0.01
)

assert np.isclose(
    customer_reporting["total_60d_merchandise_value"].sum(),
    10_606_400.87,
    atol=0.01
)

# Validate observed repeat-purchase classification
assert (
    customer_reporting["repeat_purchase_60d"].astype(bool)
    == customer_reporting["subsequent_60d_value"].gt(0)
).all()

# Validate first-order delivery classifications
assert (
    customer_reporting["first_order_delivery_status"]
    .value_counts()
    .to_dict()
    == {"Delivered": 75_599, "Not Delivered": 2_501}
)

print("FINAL CUSTOMER REPORTING DATASET VALIDATED")
print(f"Customers: {len(customer_reporting):,}")
print(f"Columns: {customer_reporting.shape[1]}")
print(f"Unique customers: {customer_reporting['customer_unique_id'].nunique():,}")

print("\nOBSERVED MONETARY VALUE")
print(
    f"Initial delivered value: "
    f"R${customer_reporting['initial_delivered_value'].sum():,.2f}"
)
print(
    f"Subsequent 60-day value: "
    f"R${customer_reporting['subsequent_60d_value'].sum():,.2f}"
)
print(
    f"Total 60-day value: "
    f"R${customer_reporting['total_60d_merchandise_value'].sum():,.2f}"
)

print("\nREPORTING SCHEMA")
display(
    pd.DataFrame({
        "column": customer_reporting.columns,
        "dtype": customer_reporting.dtypes.astype(str).values,
        "missing_values": customer_reporting.isna().sum().values
    })
)


FINAL CUSTOMER REPORTING DATASET VALIDATED
Customers: 78,100
Columns: 14
Unique customers: 78,100

OBSERVED MONETARY VALUE
Initial delivered value: R$10,395,105.68
Subsequent 60-day value: R$211,295.19
Total 60-day value: R$10,606,400.87

REPORTING SCHEMA


,column,dtype,missing_values
0,customer_unique_id,str,0
1,order_id,str,0
2,order_purchase_timestamp,datetime64[us],0
3,order_status,str,0
4,first_order_delivery_status,str,0
5,first_order_item_value,float64,642
6,repeat_purchase_propensity,float64,0
7,propensity_segment,string,0
8,initial_value_segment,string,0
9,customer_profile,string,0


In [13]:
# Export the validated customer reporting dataset

output_path = processed_dir / "customer_intelligence_reporting.parquet"

customer_reporting.to_parquet(output_path, index=False)

# Reload and verify the exported dataset
exported_reporting = pd.read_parquet(output_path)

assert len(exported_reporting) == 78_100
assert exported_reporting.shape[1] == 14
assert exported_reporting["customer_unique_id"].is_unique

pd.testing.assert_frame_equal(
    customer_reporting.reset_index(drop=True),
    exported_reporting.reset_index(drop=True)
)

print("CUSTOMER REPORTING DATASET EXPORTED SUCCESSFULLY")
print(f"File: {output_path}")
print(f"Customers: {len(exported_reporting):,}")
print(f"Columns: {exported_reporting.shape[1]}")
print("Export integrity: PASSED")

CUSTOMER REPORTING DATASET EXPORTED SUCCESSFULLY
File: c:\Users\oades\Downloads\Downloads\customer-intelligence-platform\data\processed\customer_intelligence_reporting.parquet
Customers: 78,100
Columns: 14
Export integrity: PASSED



## Business Conclusions and Analytical Limitations

### What the integration achieved

The Customer Intelligence Platform now combines repeat-purchase propensity, customer segmentation, first-order delivery status, and observed 60-day merchandise value in one validated customer-level dataset.

The reporting dataset contains 78,100 unique customers and 14 fields. Its historical monetary totals reconcile with the validated customer-value analysis.

### Key business findings

**Repeat-purchase propensity distinguishes historical purchasing behaviour.**

Customers in the high-propensity segment had an observed 60-day repeat-purchase rate of 3.58%, compared with 1.37% among low-propensity customers.

**Customer propensity and monetary value answer different business questions.**

High-propensity customers generated R$4.31 in average subsequent merchandise value, compared with R$2.19 for low-propensity customers.

However, low-propensity customers collectively generated 40.57% of subsequent merchandise value because they represent a larger customer population.

**Initial basket value and realised merchandise value are not interchangeable.**

Of the 2,501 customers whose first orders were not delivered, 1,859 had recorded first-order basket values.

The remaining 642 had missing basket values. These missing values were retained rather than replaced with zero.

**Unsuccessful first orders do not necessarily mean customers never purchase successfully.**

Among customers whose first recorded orders were not delivered, 69 subsequently generated positive merchandise value within the defined observation window.

### Analytical limitations

- Observed 60-day merchandise value is not customer lifetime value, profit, or net revenue.
- Predicted repeat-purchase propensity is not a prediction of monetary value.
- Historical segment comparisons are descriptive, not evidence of incremental revenue from targeting.
- The integrated population contains customers previously used for model development and evaluation. Results should not be presented as independent prospective validation.
- Customer value depends on the defined observation window, order eligibility rules, and delivered-order merchandise-value calculation.
- First-order basket values may exist for orders that were never delivered.

### Reporting recommendation

Use the integrated dataset to support descriptive analysis of customer segments, purchase outcomes, and observed merchandise value.

Retain experimental monetary predictions outside the main reporting dataset until their reliability and business usefulness are established through stronger validation.

### Validated output

`data/processed/customer_intelligence_reporting.parquet`

78,100 customers | 14 fields | Export integrity verified
